# NeuroState Interpretability Visualizations

   1. Boundary overlay on raw EEG (Sleep-EDF transitions)
   2. Boundary overlay on seizure onset (CHB-MIT)
   3. Regime-structured attention heatmaps
   4. Ablation summary chart
   5. t-SNE of embeddings

Loads the best ACBL + isolation v1 checkpoints for each dataset.
All model classes included inline.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import h5py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from sklearn.manifold import TSNE
from scipy.ndimage import gaussian_filter1d
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

figures_dir = Path("figures/interpretability")
figures_dir.mkdir(parents=True, exist_ok=True)

Device: cpu


In [2]:
# Publication-quality figure settings
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial'],
    'font.size': 13,
    'axes.titlesize': 15,
    'axes.titleweight': 'bold',
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'figure.titlesize': 16,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.15,
})

In [3]:
# MODEL COMPONENTS (same as all previous notebooks)

class PatchEmbedding(nn.Module):
    def __init__(self, n_channels=3, n_samples=3000, embed_dim=128,
                 temporal_kernel=25, pool_kernel=75, pool_stride=15,
                 dropout=0.1):
        super().__init__()
        self.temporal_conv = nn.Sequential(
            nn.Conv2d(1, 40, (1, temporal_kernel),
                      padding=(0, temporal_kernel // 2)),
            nn.BatchNorm2d(40), nn.GELU())
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(40, 40, (n_channels, 1)),
            nn.BatchNorm2d(40), nn.GELU())
        self.pool = nn.AvgPool2d((1, pool_kernel), stride=(1, pool_stride))
        self.projection = nn.Sequential(
            nn.Conv2d(40, embed_dim, (1, 1)), nn.Dropout(dropout))
        self.seq_len = (n_samples - pool_kernel) // pool_stride + 1
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.temporal_conv(x)
        x = self.spatial_conv(x)
        x = self.pool(x)
        x = self.projection(x)
        return x.squeeze(2).permute(0, 2, 1)

class MultiResolutionEncoder(nn.Module):
    def __init__(self, n_channels=3, n_samples=3000, embed_dim=128, dropout=0.1):
        super().__init__()
        self.enc_100 = PatchEmbedding(n_channels, n_samples, embed_dim, dropout=dropout)
        self.enc_50 = PatchEmbedding(n_channels, n_samples // 2, embed_dim, dropout=dropout)
        self.enc_25 = PatchEmbedding(n_channels, n_samples // 4, embed_dim, dropout=dropout)
        self.merge = nn.Sequential(nn.Linear(embed_dim * 3, embed_dim), nn.GELU(), nn.Dropout(dropout))
        self.seq_len_100 = self.enc_100.seq_len
    def forward(self, x):
        e100 = self.enc_100(x)
        e50 = self.enc_50(x[:, :, ::2])
        e25 = self.enc_25(x[:, :, ::4])
        T = e100.shape[1]
        e50 = F.interpolate(e50.permute(0, 2, 1), size=T, mode='linear', align_corners=False).permute(0, 2, 1)
        e25 = F.interpolate(e25.permute(0, 2, 1), size=T, mode='linear', align_corners=False).permute(0, 2, 1)
        return self.merge(torch.cat([e100, e50, e25], dim=-1))

class ContrastiveBoundaryModule(nn.Module):
    def __init__(self, embed_dim=128, hidden_dim=64, scales=(1, 4, 16), dropout=0.1):
        super().__init__()
        self.scales = scales
        self.projections = nn.ModuleList([nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim)) for _ in scales])
        self.fusion = nn.Sequential(nn.Linear(len(scales), len(scales)*2), nn.GELU(), nn.Linear(len(scales)*2, 1))
        self.temperature = nn.Parameter(torch.tensor(1.0))
    def _contrast(self, x, proj, offset):
        B, T, D = x.shape
        h = F.normalize(proj(x), dim=-1)
        if offset < T:
            h_shift = torch.roll(h, -offset, dims=1)
            h_shift[:, -offset:, :] = h[:, -offset:, :]
            sim = (h * h_shift).sum(dim=-1)
            c = 1.0 - (sim + 1.0) / 2.0
            c = torch.sigmoid((c - 0.5) * self.temperature.abs().clamp(min=0.1))
        else:
            c = torch.zeros(B, T, device=x.device)
        return c
    def forward(self, x):
        per_scale = [self._contrast(x, p, o) for p, o in zip(self.projections, self.scales)]
        stacked = torch.stack(per_scale, dim=-1)
        fused = torch.sigmoid(self.fusion(stacked).squeeze(-1))
        consistency = sum(F.mse_loss(ps, fused.detach()) for ps in per_scale) / len(per_scale)
        return {'boundaries': fused, 'per_scale': per_scale, 'boundary_loss': 0.01 * consistency}

class OriginalRegimeMask(nn.Module):
    def __init__(self): super().__init__()
    def forward(self, boundaries):
        cum = torch.cumsum(boundaries, dim=1)
        same = torch.exp(-torch.abs(cum.unsqueeze(2) - cum.unsqueeze(1)))
        return same, 1.0 - same

class RegimeStructuredAttention(nn.Module):
    def __init__(self, embed_dim=128, n_intra=4, n_inter=2, n_cross=2, dropout=0.1, regime_mask_module=None):
        super().__init__()
        self.n_heads = n_intra + n_inter + n_cross
        self.head_dim = embed_dim // self.n_heads
        self.n_intra = n_intra
        self.n_inter = n_inter
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)
        self.regime_mask = regime_mask_module or OriginalRegimeMask()
    def forward(self, x, boundaries, return_attention=False):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        same, cross = self.regime_mask(boundaries)
        same, cross = same.unsqueeze(1), cross.unsqueeze(1)
        h1, h2 = self.n_intra, self.n_intra + self.n_inter
        mask = torch.ones_like(attn)
        mask[:, :h1] = same.expand(B, self.n_intra, T, T)
        mask[:, h1:h2] = cross.expand(B, self.n_inter, T, T)
        attn = attn + torch.log(mask + 1e-6)
        w = F.softmax(attn, dim=-1)
        w = self.attn_drop(w)
        out = (w @ v).transpose(1, 2).reshape(B, T, D)
        out = self.proj_drop(self.out_proj(out))
        if return_attention: return out, w
        return out

class NeuroStateBlock(nn.Module):
    def __init__(self, embed_dim=128, n_intra=4, n_inter=2, n_cross=2,
                 mlp_ratio=4.0, dropout=0.1, regime_mask_module=None):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = RegimeStructuredAttention(embed_dim, n_intra, n_inter, n_cross, dropout, regime_mask_module)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, int(embed_dim * mlp_ratio)), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(int(embed_dim * mlp_ratio), embed_dim), nn.Dropout(dropout))
    def forward(self, x, boundaries, return_attention=False):
        if return_attention:
            attn_out, attn_w = self.attn(self.norm1(x), boundaries, return_attention=True)
            x = x + attn_out; x = x + self.mlp(self.norm2(x)); return x, attn_w
        x = x + self.attn(self.norm1(x), boundaries); x = x + self.mlp(self.norm2(x)); return x

class MultiResContrastiveNeuroState(nn.Module):
    def __init__(self, n_channels=3, n_samples=3000, n_classes=5,
                 embed_dim=128, n_layers=4, dropout=0.1,
                 contrast_scales=(1, 4, 16), cp_hidden=64, n_context_epochs=3):
        super().__init__()
        self.n_classes = n_classes; self.n_context = n_context_epochs
        self.mr_encoder = MultiResolutionEncoder(n_channels, n_samples, embed_dim, dropout)
        self.tokens_per_epoch = self.mr_encoder.seq_len_100
        total_tokens = self.tokens_per_epoch * n_context_epochs
        self.pos_embed = nn.Parameter(torch.randn(1, total_tokens, embed_dim) * 0.02)
        self.pos_drop = nn.Dropout(dropout)
        self.epoch_embed = nn.Parameter(torch.randn(1, n_context_epochs, 1, embed_dim) * 0.02)
        self.changepoint_module = ContrastiveBoundaryModule(embed_dim, cp_hidden, contrast_scales, dropout)
        self.blocks = nn.ModuleList([NeuroStateBlock(embed_dim, dropout=dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(nn.Linear(embed_dim, embed_dim // 2), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(embed_dim // 2, n_classes))
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)


def forward_full(model, x):
    """Forward pass returning boundaries, attention, embeddings, logits."""
    B, N, C, T = x.shape
    epoch_embs = []
    for i in range(N):
        emb = model.mr_encoder(x[:, i]) + model.epoch_embed[:, i]
        epoch_embs.append(emb)
    full_seq = torch.cat(epoch_embs, dim=1)
    full_seq = model.pos_drop(full_seq + model.pos_embed)
    encoder_h = full_seq
    cp_out = model.changepoint_module(full_seq)
    boundaries = cp_out['boundaries']
    all_attn = []
    for i, block in enumerate(model.blocks):
        full_seq, attn_w = block(full_seq, boundaries, return_attention=True)
        all_attn.append(attn_w)
    full_seq = model.norm(full_seq)
    tpe = model.tokens_per_epoch
    start = tpe * (N // 2)
    pooled = full_seq[:, start:start + tpe, :].mean(dim=1)
    logits = model.head(pooled)
    return {
        'logits': logits, 'boundaries': boundaries,
        'attention': all_attn, 'encoder_h': encoder_h,
        'final_h': full_seq,
    }

In [4]:
# FIGURE 1: Boundary overlay on raw EEG — Sleep-EDF transitions

def plot_boundary_overlay_sleep(model, h5_path, save_path, device):
    """Show 3-epoch windows where sleep stage transitions occur."""
    with h5py.File(h5_path, 'r') as f:
        epochs = f['epochs'][:]
        labels = f['labels'][:]
        subject_ids = np.array([s.decode() if isinstance(s, bytes) else str(s)
                                for s in f['subject_ids'][:]])

    stage_names = ['Wake', 'N1', 'N2', 'N3', 'REM']
    colors_stage = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12']
    sfreq = 100
    tpe = model.tokens_per_epoch
    spt = 30.0 / tpe

    # Find windows containing transitions
    transition_windows = []
    for i in range(1, len(labels) - 1):
        if subject_ids[i-1] == subject_ids[i] == subject_ids[i+1]:
            if labels[i-1] != labels[i] or labels[i] != labels[i+1]:
                transition_windows.append(i)

    np.random.seed(42)
    selected = np.random.choice(transition_windows,
                                size=min(4, len(transition_windows)),
                                replace=False)
    n_plots = len(selected)

    fig, axes = plt.subplots(n_plots, 2, figsize=(15, 3.2 * n_plots),
                             gridspec_kw={'width_ratios': [1, 1],
                                          'hspace': 0.40, 'wspace': 0.18})
    if n_plots == 1:
        axes = axes.reshape(1, -1)

    model.eval()
    legend_added = False

    for row, center_idx in enumerate(selected):
        # Build 3-epoch window
        eps = []
        for off in [-1, 0, 1]:
            ep = epochs[center_idx + off].astype(np.float32)
            nc = ep.shape[0]
            if nc < 3:
                ep = np.vstack([ep, np.zeros((3 - nc, ep.shape[1]),
                                             dtype=np.float32)])
            eps.append(ep)
        x = torch.tensor(np.stack(eps),
                         dtype=torch.float32).unsqueeze(0).to(device)
        el = [int(labels[center_idx + off]) for off in [-1, 0, 1]]

        with torch.no_grad():
            out = forward_full(model, x)
        bnd = out['boundaries'][0].cpu().numpy()
        bnd_smooth = gaussian_filter1d(bnd, sigma=3)

        raw_signal = np.concatenate([
            epochs[center_idx + off][0] for off in [-1, 0, 1]])
        time_eeg = np.arange(len(raw_signal)) / sfreq
        time_bnd = np.arange(len(bnd)) * spt

        # ===== Left panel: raw EEG =====
        ax = axes[row, 0]
        ax.plot(time_eeg, raw_signal * 1e6, 'k-', linewidth=0.4, alpha=0.85)
        for ep_idx, off in enumerate([-1, 0, 1]):
            t_start = ep_idx * 30
            t_end = (ep_idx + 1) * 30
            stage = el[ep_idx]
            ax.axvspan(t_start, t_end, alpha=0.15, color=colors_stage[stage])

        # Lock stage labels to top of panel after data is plotted
        ymin, ymax = ax.get_ylim()
        for ep_idx in range(3):
            t_mid = ep_idx * 30 + 15
            stage = el[ep_idx]
            ax.text(t_mid, ymax * 0.95, stage_names[stage],
                    ha='center', va='top', fontsize=12, fontweight='bold',
                    color=colors_stage[stage])

        ax.axvline(30, color='gray', linestyle='--', alpha=0.5, linewidth=1.2)
        ax.axvline(60, color='gray', linestyle='--', alpha=0.5, linewidth=1.2)
        ax.set_ylabel('EEG (μV)', fontsize=13)
        ax.tick_params(axis='both', labelsize=11)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        if row == n_plots - 1:
            ax.set_xlabel('Time (seconds)', fontsize=13)
        if row == 0:
            ax.set_title('Raw EEG signal (Fpz-Cz)',
                         fontsize=15, fontweight='bold', pad=10)

        # ===== Right panel: boundary activation =====
        ax2 = axes[row, 1]
        ax2.fill_between(time_bnd, 0, bnd_smooth, alpha=0.3, color='#A32638')
        ax2.plot(time_bnd, bnd_smooth, color='#A32638', linewidth=1.8)
        ax2.axvline(30, color='gray', linestyle='--', alpha=0.5, linewidth=1.2)
        ax2.axvline(60, color='gray', linestyle='--', alpha=0.5, linewidth=1.2)
        ax2.set_ylabel('Boundary\nactivation', fontsize=13)
        ax2.set_ylim(0, max(bnd_smooth.max() * 1.2, 0.05))
        ax2.tick_params(axis='both', labelsize=11)
        ax2.spines['top'].set_visible(False)
        ax2.spines['right'].set_visible(False)
        if row == n_plots - 1:
            ax2.set_xlabel('Time (seconds)', fontsize=13)
        if row == 0:
            ax2.set_title('Learned boundary probability',
                          fontsize=15, fontweight='bold', pad=10)

        # Mark true transitions with green vertical line
        for ep_idx in range(2):
            if el[ep_idx] != el[ep_idx + 1]:
                t_trans = (ep_idx + 1) * 30
                label = 'True transition' if not legend_added else None
                ax2.axvline(t_trans, color='#28a745', linewidth=2.5,
                            alpha=0.85, label=label)
                if label is not None:
                    legend_added = True

    if legend_added:
        axes[0, 1].legend(fontsize=12, loc='upper right',
                          frameon=True, framealpha=0.95)

    plt.suptitle('Boundary Activation at Sleep Stage Transitions',
                 fontsize=18, fontweight='bold', y=1.00)
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {save_path}")

In [5]:
# FIGURE 2: Boundary overlay on seizure onset — CHB-MIT

def plot_boundary_overlay_seizure(model, h5_path, save_path, device):
    """Show boundary activation around seizure onset."""
    with h5py.File(h5_path, 'r') as f:
        epochs = f['epochs'][:]
        labels = f['labels'][:]
        subject_ids = np.array([s.decode() for s in f['subject_ids'][:]])

    sfreq = 100
    tpe = model.tokens_per_epoch
    spt = 30.0 / tpe

    onset_windows = []
    for i in range(1, len(labels) - 1):
        if subject_ids[i-1] == subject_ids[i] == subject_ids[i+1]:
            if labels[i-1] == 0 and labels[i] == 1:
                onset_windows.append(i)

    np.random.seed(42)
    selected = onset_windows[:min(4, len(onset_windows))]
    if len(selected) == 0:
        seizure_idx = np.where(labels == 1)[0]
        selected = []
        for idx in seizure_idx[:4]:
            if 0 < idx < len(labels) - 1 and \
               subject_ids[idx-1] == subject_ids[idx] == subject_ids[idx+1]:
                selected.append(idx)

    n_plots = min(4, len(selected))
    if n_plots == 0:
        print("No suitable seizure windows found")
        return

    fig, axes = plt.subplots(n_plots, 2, figsize=(15, 3.2 * n_plots),
                             gridspec_kw={'width_ratios': [1, 1],
                                          'hspace': 0.40, 'wspace': 0.18})
    if n_plots == 1:
        axes = axes.reshape(1, -1)

    model.eval()
    legend_added = False

    for row, center_idx in enumerate(selected):
        eps = [epochs[center_idx + off].astype(np.float32)
               for off in [-1, 0, 1]]
        x = torch.tensor(np.stack(eps),
                         dtype=torch.float32).unsqueeze(0).to(device)
        el = [int(labels[center_idx + off]) for off in [-1, 0, 1]]

        with torch.no_grad():
            out = forward_full(model, x)
        bnd = out['boundaries'][0].cpu().numpy()
        bnd_smooth = gaussian_filter1d(bnd, sigma=3)

        raw_signal = np.concatenate([
            epochs[center_idx + off].mean(axis=0) for off in [-1, 0, 1]])
        time_eeg = np.arange(len(raw_signal)) / sfreq
        time_bnd = np.arange(len(bnd)) * spt

        # ===== Left panel: raw EEG =====
        ax = axes[row, 0]
        ax.plot(time_eeg, raw_signal, 'k-', linewidth=0.4, alpha=0.85)
        for ep_idx in range(3):
            t_s, t_e = ep_idx * 30, (ep_idx + 1) * 30
            color = '#e74c3c' if el[ep_idx] == 1 else '#3498db'
            ax.axvspan(t_s, t_e, alpha=0.15, color=color)

        # Lock labels to top of panel
        ymin, ymax = ax.get_ylim()
        for ep_idx in range(3):
            t_mid = ep_idx * 30 + 15
            color = '#e74c3c' if el[ep_idx] == 1 else '#3498db'
            label = 'SEIZURE' if el[ep_idx] == 1 else 'Normal'
            ax.text(t_mid, ymax * 0.95, label,
                    ha='center', va='top', fontsize=12, fontweight='bold',
                    color=color)

        ax.axvline(30, color='gray', linestyle='--', alpha=0.5, linewidth=1.2)
        ax.axvline(60, color='gray', linestyle='--', alpha=0.5, linewidth=1.2)
        ax.set_ylabel('EEG (mean ch)', fontsize=13)
        ax.tick_params(axis='both', labelsize=11)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        if row == n_plots - 1:
            ax.set_xlabel('Time (seconds)', fontsize=13)
        if row == 0:
            ax.set_title('Raw EEG signal',
                         fontsize=15, fontweight='bold', pad=10)

        # ===== Right panel: boundary =====
        ax2 = axes[row, 1]
        ax2.fill_between(time_bnd, 0, bnd_smooth, alpha=0.3, color='#A32638')
        ax2.plot(time_bnd, bnd_smooth, color='#A32638', linewidth=1.8)
        ax2.axvline(30, color='gray', linestyle='--', alpha=0.5, linewidth=1.2)
        ax2.axvline(60, color='gray', linestyle='--', alpha=0.5, linewidth=1.2)

        # Mark onsets — only first one in legend
        for ep_idx in range(2):
            if el[ep_idx] != el[ep_idx + 1]:
                label = 'Onset' if not legend_added else None
                ax2.axvline((ep_idx + 1) * 30, color='#28a745',
                            linewidth=2.5, alpha=0.85, label=label)
                if label is not None:
                    legend_added = True

        ax2.set_ylabel('Boundary\nactivation', fontsize=13)
        ax2.set_ylim(0, max(bnd_smooth.max() * 1.2, 0.05))
        ax2.tick_params(axis='both', labelsize=11)
        ax2.spines['top'].set_visible(False)
        ax2.spines['right'].set_visible(False)
        if row == n_plots - 1:
            ax2.set_xlabel('Time (seconds)', fontsize=13)
        if row == 0:
            ax2.set_title('Learned boundary probability',
                          fontsize=15, fontweight='bold', pad=10)

    if legend_added:
        axes[0, 1].legend(fontsize=12, loc='upper right',
                          frameon=True, framealpha=0.95)

    plt.suptitle('Boundary Activation at Seizure Onset',
                 fontsize=18, fontweight='bold', y=1.00)
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {save_path}")

In [6]:
# FIGURE 3: Attention heatmaps — intra vs inter vs cross-scale

def plot_attention_heatmaps(model, h5_path, save_path, device, n_channels=3):
    """Show how the 3 head groups attend differently."""
    with h5py.File(h5_path, 'r') as f:
        epochs = f['epochs'][:]
        labels = f['labels'][:]
        subject_ids = np.array([s.decode() if isinstance(s, bytes) else str(s)
                                for s in f['subject_ids'][:]])

    # Find a transition window
    center_idx = None
    for i in range(1, len(labels) - 1):
        if subject_ids[i-1] == subject_ids[i] == subject_ids[i+1]:
            if labels[i-1] != labels[i]:
                center_idx = i
                break
    if center_idx is None:
        print("No transition window found")
        return

    eps = []
    for off in [-1, 0, 1]:
        ep = epochs[center_idx + off].astype(np.float32)
        nc = ep.shape[0]
        if nc < n_channels:
            ep = np.vstack([ep, np.zeros((n_channels - nc, ep.shape[1]),
                                         dtype=np.float32)])
        eps.append(ep)
    x = torch.tensor(np.stack(eps),
                     dtype=torch.float32).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        out = forward_full(model, x)

    attn = out['attention'][-1][0].cpu().numpy()
    intra_attn = attn[:4].mean(axis=0)
    inter_attn = attn[4:6].mean(axis=0)
    cross_attn = attn[6:8].mean(axis=0)

    step = 4
    tpe = model.tokens_per_epoch

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    titles = ['Intra-regime heads (0-3)',
              'Inter-regime heads (4-5)',
              'Cross-scale heads (6-7)']
    data = [intra_attn, inter_attn, cross_attn]
    cmaps = ['Blues', 'Reds', 'Greens']

    for ax, title, d, cmap in zip(axes, titles, data, cmaps):
        d_sub = d[::step, ::step]
        im = ax.imshow(d_sub, aspect='auto', cmap=cmap, interpolation='nearest')
        # Mark epoch boundaries
        for ep in range(1, 3):
            pos = (ep * tpe) / step
            ax.axhline(pos, color='white', linewidth=1.2, alpha=0.85)
            ax.axvline(pos, color='white', linewidth=1.2, alpha=0.85)
        ax.set_title(title, fontsize=15, fontweight='bold', pad=10)
        ax.set_xlabel('Key token', fontsize=13, labelpad=6)
        ax.set_ylabel('Query token', fontsize=13, labelpad=6)
        ax.tick_params(axis='both', labelsize=11)
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=10)

    el = [int(labels[center_idx + off]) for off in [-1, 0, 1]]
    stage_names = ['Wake', 'N1', 'N2', 'N3', 'REM']
    transition_str = ' → '.join([stage_names[s] for s in el])
    plt.suptitle(f'Regime-Structured Attention Patterns ({transition_str})',
                 fontsize=17, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {save_path}")

In [7]:
# FIGURE 4: Ablation summary bar chart

def plot_ablation_chart(save_path):
    """Clean bar chart comparing ablation results."""
    labels = ['Full model\n(isolation)',
                'No pseudo-\nboundary',
                'No Gaussian\ntargets',
                'No attention\nKL prior',
                'No gradient\nisolation']

    sleep_acc = [0.718, 0.719, 0.706, 0.703, 0.698]
    sleep_bnd = [0.028, 0.0052, 0.0001, 0.0052, 0.0048]
    chb_auroc = [0.940, 0.936, 0.931, 0.927, 0.929]
    chb_bnd = [0.035, 0.019, 0.001, 0.015, 0.030]

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    x = np.arange(len(labels))
    w = 0.65
    colors = ['#28a745', '#6c757d', '#dc3545', '#6c757d', '#6c757d']

    def style_axis(ax):
        ax.tick_params(axis='y', labelsize=12)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    # Sleep-EDF Accuracy
    ax = axes[0, 0]
    bars = ax.bar(x, sleep_acc, w, color=colors, alpha=0.9,
                  edgecolor='black', linewidth=0.8)
    ax.set_ylabel('Accuracy', fontsize=14)
    ax.set_title('Sleep-EDF — Classification',
                 fontsize=15, fontweight='bold', pad=8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylim(0.68, 0.73)
    for bar, val in zip(bars, sleep_acc):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.3f}', ha='center', fontsize=12, fontweight='bold')
    style_axis(ax)

    # Sleep-EDF Boundary Std
    ax = axes[0, 1]
    bars = ax.bar(x, sleep_bnd, w, color=colors, alpha=0.9,
                  edgecolor='black', linewidth=0.8)
    ax.set_ylabel('Boundary Std', fontsize=14)
    ax.set_title('Sleep-EDF — Boundary Structure',
                 fontsize=15, fontweight='bold', pad=8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylim(0, max(sleep_bnd) * 1.18)
    for bar, val in zip(bars, sleep_bnd):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(sleep_bnd) * 0.02,
                f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')
    style_axis(ax)

    # CHB-MIT AUROC
    ax = axes[1, 0]
    bars = ax.bar(x, chb_auroc, w, color=colors, alpha=0.9,
                  edgecolor='black', linewidth=0.8)
    ax.set_ylabel('AUROC', fontsize=14)
    ax.set_title('CHB-MIT — Classification',
                 fontsize=15, fontweight='bold', pad=8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylim(0.92, 0.945)
    for bar, val in zip(bars, chb_auroc):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0004,
                f'{val:.3f}', ha='center', fontsize=12, fontweight='bold')
    style_axis(ax)

    # CHB-MIT Boundary Std
    ax = axes[1, 1]
    bars = ax.bar(x, chb_bnd, w, color=colors, alpha=0.9,
                  edgecolor='black', linewidth=0.8)
    ax.set_ylabel('Boundary Std', fontsize=14)
    ax.set_title('CHB-MIT — Boundary Structure',
                 fontsize=15, fontweight='bold', pad=8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylim(0, max(chb_bnd) * 1.18)
    for bar, val in zip(bars, chb_bnd):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(chb_bnd) * 0.02,
                f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')
    style_axis(ax)

    plt.suptitle('Ablation Study: Component Contributions to ACBL',
                 fontsize=18, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {save_path}")

In [8]:
# FIGURE 5: t-SNE of token embeddings colored by sleep stage

def plot_tsne_embeddings(model, h5_path, save_path, device):
    """t-SNE of center-epoch pooled embeddings, colored by class."""
    with h5py.File(h5_path, 'r') as f:
        epochs = f['epochs'][:]
        labels = f['labels'][:]
        subject_ids = np.array([s.decode() if isinstance(s, bytes) else str(s)
                                for s in f['subject_ids'][:]])

    stage_names = ['Wake', 'N1', 'N2', 'N3', 'REM']
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12']

    unique_subj = np.unique(subject_ids)
    np.random.seed(42)
    np.random.shuffle(unique_subj)
    test_subj = set(unique_subj[int(0.85 * len(unique_subj)):])

    all_embs, all_labels = [], []
    model.eval()

    for i in range(1, len(labels) - 1):
        if subject_ids[i] not in test_subj:
            continue
        if subject_ids[i-1] != subject_ids[i] or \
           subject_ids[i] != subject_ids[i+1]:
            continue

        eps = []
        for off in [-1, 0, 1]:
            ep = epochs[i + off].astype(np.float32)
            nc = ep.shape[0]
            if nc < 3:
                ep = np.vstack([ep, np.zeros((3 - nc, ep.shape[1]),
                                             dtype=np.float32)])
            eps.append(ep)
        x = torch.tensor(np.stack(eps),
                         dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            out = forward_full(model, x)
            tpe = model.tokens_per_epoch
            start = tpe
            emb = out['final_h'][0, start:start + tpe, :].mean(dim=0).cpu().numpy()
        all_embs.append(emb)
        all_labels.append(int(labels[i]))

        if len(all_embs) >= 1000:
            break

    all_embs = np.array(all_embs)
    all_labels = np.array(all_labels)
    print(f"t-SNE on {len(all_embs)} samples")

    tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
    coords = tsne.fit_transform(all_embs)

    fig, ax = plt.subplots(figsize=(11, 9))
    for c in range(5):
        mask = all_labels == c
        if mask.any():
            ax.scatter(coords[mask, 0], coords[mask, 1],
                       c=colors[c], label=stage_names[c],
                       s=28, alpha=0.7, edgecolors='white', linewidth=0.3)

    legend = ax.legend(fontsize=14, markerscale=2.0,
                       loc='upper right', frameon=True, framealpha=0.95,
                       edgecolor='gray')
    ax.set_title('t-SNE of Center-Epoch Embeddings (Sleep-EDF)',
                 fontsize=18, fontweight='bold', pad=14)
    ax.set_xlabel('t-SNE 1', fontsize=15, labelpad=8)
    ax.set_ylabel('t-SNE 2', fontsize=15, labelpad=8)
    ax.tick_params(axis='both', labelsize=12)
    ax.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {save_path}")

In [9]:
# MAIN

def main():
    print("=" * 60)
    print("NEUROSTATE INTERPRETABILITY VISUALIZATIONS")
    print("=" * 60)

    # Sleep-EDF visualizations
    sleep_h5 = Path("data/processed/sleep_edf_processed.h5")
    sleep_ckpt = Path("models/acbl_isolation_chbmit_best.pt")
    # Try to find the sleep staging checkpoint
    for p in [Path("models/neurostate_acbl_isolated.pt")]:
        if p.exists():
            sleep_ckpt = p; break

    if sleep_h5.exists():
        print("\n Loading Sleep-EDF model")
        model_sleep = MultiResContrastiveNeuroState(
            n_channels=3, n_samples=3000, n_classes=5,
            embed_dim=128, n_layers=4, dropout=0.1,
            contrast_scales=(1, 4, 16), cp_hidden=64,
            n_context_epochs=3).to(device)

        # Try loading checkpoint; if not found, use random init (for structure only)
        if sleep_ckpt.exists():
            ckpt = torch.load(sleep_ckpt, map_location=device, weights_only=False)
            state = ckpt['model_state_dict']
            fixed = {}
            for k, v in state.items():
                k = k.replace('enc_100hz', 'enc_100').replace('enc_50hz', 'enc_50').replace('enc_25hz', 'enc_25')
                fixed[k] = v
            model_sleep.load_state_dict(fixed, strict=False)
            print(f"  Loaded: {sleep_ckpt}")
        else:
            print(f"  Warning: No Sleep-EDF checkpoint found, using random init")
            print(f"  Looked for: {sleep_ckpt}")
            print(f"  Figures will show architecture behavior, not trained results")

        print("\nFigure 1: Boundary overlay on sleep transitions")
        plot_boundary_overlay_sleep(model_sleep, sleep_h5,
                                    figures_dir / 'boundary_overlay_sleep.png', device)

        print("\nFigure 3: Attention heatmaps")
        plot_attention_heatmaps(model_sleep, sleep_h5,
                                figures_dir / 'attention_heatmaps.png', device, n_channels=3)

        print("\nFigure 5: t-SNE embeddings")
        plot_tsne_embeddings(model_sleep, sleep_h5,
                             figures_dir / 'tsne_embeddings_sleep.png', device)

    # CHB-MIT visualizations
    chb_h5 = Path("data/processed/chbmit_combined_seizure_detection.h5")
    chb_ckpt = Path("models/acbl_isolation_chbmit_best.pt")

    if chb_h5.exists() and chb_ckpt.exists():
        print("\n Loading CHB-MIT model")
        model_chb = MultiResContrastiveNeuroState(
            n_channels=17, n_samples=3000, n_classes=2,
            embed_dim=128, n_layers=4, dropout=0.1,
            contrast_scales=(1, 4, 16), cp_hidden=64,
            n_context_epochs=3).to(device)
        ckpt = torch.load(chb_ckpt, map_location=device, weights_only=False)
        model_chb.load_state_dict(ckpt['model_state_dict'])
        print(f"  Loaded: {chb_ckpt}")

        print("\nFigure 2: Boundary overlay on seizure onset")
        plot_boundary_overlay_seizure(model_chb, chb_h5,
                                      figures_dir / 'boundary_overlay_seizure.png', device)

    # Ablation chart (no model needed)
    print("\nFigure 4: Ablation summary chart")
    plot_ablation_chart(figures_dir / 'ablation_summary_chart.png')

    print(f"\nAll figures saved to {figures_dir}/")


if __name__ == "__main__":
    main()

NEUROSTATE INTERPRETABILITY VISUALIZATIONS

 Loading Sleep-EDF model
  Loaded: models/neurostate_acbl_isolated.pt

Figure 1: Boundary overlay on sleep transitions
Saved: figures/interpretability/boundary_overlay_sleep.png

Figure 3: Attention heatmaps
Saved: figures/interpretability/attention_heatmaps.png

Figure 5: t-SNE embeddings
t-SNE on 1000 samples
Saved: figures/interpretability/tsne_embeddings_sleep.png

 Loading CHB-MIT model
  Loaded: models/acbl_isolation_chbmit_best.pt

Figure 2: Boundary overlay on seizure onset
Saved: figures/interpretability/boundary_overlay_seizure.png

Figure 4: Ablation summary chart
Saved: figures/interpretability/ablation_summary_chart.png

All figures saved to figures/interpretability/


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=30d601c2-0a51-44b0-ac6c-72cf48d1679e' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>